# Survivorship-free + point-in-time Indian equity data — the sample in use

Most Indian equity backtests are quietly inflated by two bugs that live in the **data**, not the strategy:
**survivorship bias** (delisted names dropped) and **look-ahead bias** (restated fundamentals).
This notebook runs on the real sample CSVs and shows both — on names you can verify.

In [ ]:
import glob, os
import pandas as pd
import matplotlib.pyplot as plt

# Load the sample CSVs from wherever this runs: Kaggle mount, a local clone, or
# (on Colab) straight from the GitHub repo. pandas reads https URLs directly.
RAW = 'https://raw.githubusercontent.com/Finance-broski/pit-data-sample/main/sample_data'
def find(name):
    for pat in (f'/kaggle/input/**/{name}', f'sample_data/{name}', name):
        hits = glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    return f'{RAW}/{name}'   # Colab / anywhere with internet

uni  = pd.read_csv(find('survivorship_universe.csv'))
fund = pd.read_csv(find('fundamentals_sample.csv'), parse_dates=['period_end', 'announce_date'])
px   = pd.read_csv(find('prices_sample.csv'), parse_dates=['date'])
print({k: v.shape for k, v in {'universe': uni, 'fundamentals': fund, 'prices': px}.items()})

## 1. Survivorship — the names your screener silently drops

A survivorship-free universe keeps companies that delisted, merged, or went to zero, on the dates they actually traded.

In [ ]:
dead = uni[uni.status == 'inactive']
print(f"{len(dead)} delisted/suspended names out of {len(uni)} total\n")
famous = ['DHFL','JETAIRWAYS','RCOM','RELCAPITAL','IL&FSENGG','GITANJALI','RUCHISOYA','ANDHRABANK']
print(dead[dead.symbol.isin(famous)][['symbol','company_name','first_traded','last_traded']].to_string(index=False))

## 2. Point-in-time fundamentals — read DHFL's collapse as it was announced

Every figure is stamped with its **announcement date** (`announce_date`) — the day it became public, not a
restated value. `announce_lag_days` is the gap a restated screener erases (and how look-ahead bias sneaks in).

In [ ]:
dhfl = fund[fund.symbol == 'DHFL'].sort_values('period_end')
print(dhfl[['period_end','announce_date','revenue_cr','pat_cr','announce_lag_days']].to_string(index=False))
print(f"\nmedian announcement lag across the sample: {fund.announce_lag_days.median():.0f} days")
assert (fund.announce_lag_days > 0).all(), "every figure is dated AFTER its period end — no look-ahead"
print("check passed: 0 figures dated before their period end (no look-ahead).")

## 3. Survivorship made visible — a delisted stock plotted to zero

A survivor-only dataset would have *removed* this line entirely, so your backtest would never have held it.

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
for sym, color in [('DHFL','#cc2b2b'), ('RELIANCE','#1f4e9c')]:
    s = px[px.symbol == sym].sort_values('date')
    if len(s):
        ax.plot(s.date, s.tr_close / s.tr_close.iloc[0] * 100, label=sym, color=color, lw=2)
ax.set_yscale('log'); ax.set_ylabel('Total-return index (start = 100, log)')
ax.set_title('DHFL (delisted, → 0) vs a survivor — the line survivorship bias deletes')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Takeaway

- The universe keeps **1,209 delisted names** other sources drop → no survivorship bias.
- Fundamentals are **announce-dated** → no look-ahead.
- Prices are corporate-action-adjusted with a total-return series.

This is a **bounded free sample**. Full dataset (~3,800 names of prices, ~1,400 names of as-reported fundamentals with
full vintages, point-in-time universe membership): **https://github.com/Finance-broski/pit-data-sample** — or open an
issue to get it run on your own tickers.